# TP53 expression in TCGA-BRCA

This notebook is a public, reproducible version of the Claude Science local
Qwen demo. It downloads the TCGA-BRCA Xena expression matrix, extracts TP53,
compares primary tumor and normal samples, and writes the same plot/summary
artifact shape used in the demo GIF.


In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

URL = "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/HiSeqV2.gz"
DATA = Path("TCGA.BRCA.HiSeqV2.gz")

if not DATA.exists():
    urllib.request.urlretrieve(URL, DATA)

print(f"Matrix ready: {DATA} ({DATA.stat().st_size:,} bytes)")

In [ ]:
expr = pd.read_csv(DATA, sep="\t", low_memory=False)
tp53_row = expr.loc[expr["sample"].eq("TP53")].iloc[0]

samples = pd.Index(expr.columns[1:], name="sample")
values = pd.to_numeric(tp53_row.iloc[1:], errors="coerce")
sample_type = samples.to_series().str.split("-").str[3].str[:2]

tp53 = pd.DataFrame({
    "sample": samples,
    "TP53_log2_TPM_plus_1": values.to_numpy(),
    "sample_type": sample_type.to_numpy(),
}).dropna()

normal = tp53.loc[tp53["sample_type"].eq("11"), "TP53_log2_TPM_plus_1"]
tumor = tp53.loc[tp53["sample_type"].eq("01"), "TP53_log2_TPM_plus_1"]
welch = stats.ttest_ind(tumor, normal, equal_var=False)

summary = pd.DataFrame({
    "group": ["Normal (-11)", "Primary tumor (-01)"],
    "n": [len(normal), len(tumor)],
    "mean": [normal.mean(), tumor.mean()],
    "median": [normal.median(), tumor.median()],
    "sd": [normal.std(), tumor.std()],
})

delta = tumor.mean() - normal.mean()
summary

In [ ]:
rng = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(7, 5.5))

bp = ax.boxplot(
    [normal, tumor],
    widths=0.4,
    patch_artist=True,
    labels=[f"Normal (n={len(normal)})", f"Primary tumor (n={len(tumor)})"],
    medianprops={"color": "#e41a1c", "linewidth": 2},
    boxprops={"edgecolor": "#333"},
    whiskerprops={"color": "#333"},
    capprops={"color": "#333"},
)
bp["boxes"][0].set_facecolor("#d8e8d8")
bp["boxes"][1].set_facecolor("#e8d8d8")

def jitter(x, n, scale=0.08):
    return x + rng.uniform(-scale, scale, size=n)

ax.scatter(jitter(1, len(normal)), normal, s=10, alpha=0.35, color="#2b8c3e")
ax.scatter(jitter(2, len(tumor)), tumor, s=10, alpha=0.35, color="#b22222")
ax.set_title("TP53 differential expression in TCGA-BRCA", weight="bold")
ax.set_ylabel("TP53 expression (log2 TPM + 1)")
ax.text(1.5, max(tumor.max(), normal.max()) - 0.15, f"Delta = {delta:+.02f}", ha="center")
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig.savefig("tp53_expression_plot.png", dpi=200)

summary_md = f"""# TP53 differential expression in TCGA-BRCA

- Normal samples: n={len(normal)}, mean={normal.mean():.4f}, median={normal.median():.4f}
- Primary tumor samples: n={len(tumor)}, mean={tumor.mean():.4f}, median={tumor.median():.4f}
- Mean difference, tumor - normal: {delta:+.4f}
- Welch t-test: t={welch.statistic:.3f}, p={welch.pvalue:.3g}

Interpretation: TP53 mRNA abundance is very similar between TCGA-BRCA
primary tumors and normal adjacent samples in this matrix.
"""
Path("tp53_summary.md").write_text(summary_md)
print("Wrote tp53_expression_plot.png and tp53_summary.md")